In [1]:
from typing import final

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

#reading data
df_full = pd.read_csv("007_manchester_processed.csv")

#sort
df_full = df_full.sort_values(by='time').reset_index(drop=True)

# choose training and test dataset
df = df_full[(df_full['year'] >= 2006) & (df_full['year'] <= 2050)].copy()


df = df.drop(columns=['lat', 'lon', 'date'], errors='ignore')

print("data shape：", df.shape)
print(df.head())

data shape： (15841, 14)
         time  TREFMXAV_U       FLNS       FSNS         PRECT          PRSN  \
0  2006-01-02     8.91305  32.912292   9.570811  1.024123e-08  1.328980e-15   
1  2006-01-03    10.54960  27.826134  18.303064  1.065816e-08  4.577285e-16   
2  2006-01-04     8.91406  65.338554  28.210258  2.143026e-09  7.480785e-16   
3  2006-01-05     7.02330  38.994473  16.601635  4.274406e-08  1.316970e-14   
4  2006-01-06     6.15830   7.846845   3.547717  4.993412e-08  3.063486e-09   

       QBOT   TREFHT      UBOT      VBOT  year  month  day  dayofyear  
0  0.003757  6.76428  8.547442  0.224397  2006      1    2          2  
1  0.004989  6.60630  3.954805  1.289937  2006      1    3          3  
2  0.004033  4.84060  6.494915 -0.385829  2006      1    4          4  
3  0.004264  3.19866 -0.893610  0.834761  2006      1    5          5  
4  0.004556  4.12690 -4.548933  0.594513  2006      1    6          6  


In [2]:
target = 'TREFMXAV_U'

features = [col for col in df.columns if col not in [target, 'time']]

X_all = df[features].values
y_all = df[target].values

In [3]:

train_years = 5
val_years = 3
stride_years = 5

days_per_year = 365

train_size = train_years * days_per_year
val_size = val_years * days_per_year
stride_size = stride_years * days_per_year


def sliding_split(X, y, df, train_size, val_size, stride_size):
    splits = []

    start = 0
    total = len(X)

    while start + train_size + val_size <= total:

        train_end = start + train_size
        val_end = train_end + val_size

        X_train = X[start:train_end]
        y_train = y[start:train_end]

        X_val = X[train_end:val_end]
        y_val = y[train_end:val_end]

        # record time
        train_start_year = df.iloc[start]['year']
        train_end_year = df.iloc[train_end - 1]['year']

        val_start_year = df.iloc[train_end]['year']
        val_end_year = df.iloc[val_end - 1]['year']

        splits.append((
            X_train, y_train, X_val, y_val,
            train_start_year, train_end_year,
            val_start_year, val_end_year
        ))

        start += stride_size

    return splits


splits = sliding_split(X_all, y_all, df, train_size, val_size, stride_size)

print("Windows：", len(splits))

Windows： 8


In [4]:
mae_list = []
rmse_list = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")
    print(f"Train size: {len(X_train)} | Val size: {len(X_val)}")

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    mae_list.append(mae)
    rmse_list.append(rmse)

    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")

print("\n===== result =====")
print("Average MAE:", np.mean(mae_list))
print("Average RMSE:", np.mean(rmse_list))


===== Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
Train size: 1825 | Val size: 1095
MAE: 0.603, RMSE: 0.797

===== Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
Train size: 1825 | Val size: 1095
MAE: 0.581, RMSE: 0.761

===== Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
Train size: 1825 | Val size: 1095
MAE: 0.598, RMSE: 0.799

===== Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
Train size: 1825 | Val size: 1095
MAE: 0.609, RMSE: 0.800

===== Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2034
Train size: 1825 | Val size: 1095
MAE: 0.618, RMSE: 0.799

===== Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
Train size: 1825 | Val size: 1095
MAE: 0.633, RMSE: 0.845

===== Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
Train size: 1825 | Val size: 1095
MAE: 0.631, RMSE: 0.839

===== Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
Train size: 1825 | Val size: 1095
MAE: 0.622, RMSE: 0.816

===== result =====
Average MAE: 0.6119580451344304
Average RMSE

In [5]:
final_train_df = df_full[df_full['year'] <= 2050]
final_test_df  = df_full[df_full['year'] > 2050]
X_final_train = final_train_df[features].values
y_final_train = final_train_df[target].values

X_final_test = final_test_df[features].values
y_final_test = final_test_df[target].values


In [6]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_final_train, y_final_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [7]:
from sklearn.metrics import r2_score, max_error

y_pred = rf.predict(X_final_test)

rf_mae = mean_absolute_error(y_final_test, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
rf_r2 = r2_score(y_final_test, y_pred)
rf_max_err = max_error(y_final_test, y_pred)

print("===== Test Results =====")
print(f"MAE: {rf_mae:.3f}")
print(f"RMSE: {rf_rmse:.3f}")
print(f"R2: {rf_r2:.3f}")
print(f"Max Error: {rf_max_err:.3f}")

===== Test Results =====
MAE: 0.575
RMSE: 0.760
R2: 0.981
Max Error: 4.980


In [8]:
from xgboost import XGBRegressor
mae_list_xgb = []
rmse_list_xgb = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== XGB Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))


    mae_list_xgb.append(mae)
    rmse_list_xgb.append(rmse)

    print(f"XGB MAE: {mae:.3f}, RMSE: {rmse:.3f}")

print("\n===== XGBoost result =====")
print("Average MAE:", np.mean(mae_list_xgb))
print("Average RMSE:", np.mean(rmse_list_xgb))


===== XGB Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
XGB MAE: 0.560, RMSE: 0.760

===== XGB Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
XGB MAE: 0.532, RMSE: 0.710

===== XGB Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
XGB MAE: 0.563, RMSE: 0.763

===== XGB Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
XGB MAE: 0.547, RMSE: 0.722

===== XGB Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2034
XGB MAE: 0.553, RMSE: 0.732

===== XGB Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
XGB MAE: 0.569, RMSE: 0.768

===== XGB Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
XGB MAE: 0.589, RMSE: 0.779

===== XGB Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
XGB MAE: 0.560, RMSE: 0.739

===== XGBoost result =====
Average MAE: 0.5593190239889048
Average RMSE: 0.7464984175719704


In [9]:
xgb_final = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_final.fit(X_final_train, y_final_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [10]:
from sklearn.metrics import r2_score, max_error

y_pred = xgb_final.predict(X_final_test)

xgb_mae = mean_absolute_error(y_final_test, y_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
xgb_r2 = r2_score(y_final_test, y_pred)
xgb_max_err = max_error(y_final_test, y_pred)

print("\n===== XGBoost Test Results =====")
print(f"MAE: {xgb_mae:.3f}")
print(f"RMSE: {xgb_rmse:.3f}")
print(f"R2: {xgb_r2:.3f}")
print(f"Max Error: {xgb_max_err:.3f}")


===== XGBoost Test Results =====
MAE: 0.530
RMSE: 0.710
R2: 0.983
Max Error: 5.126


In [11]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_list_lgb = []
rmse_list_lgb = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== LGBM Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")

    lgb = LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )


    lgb.fit(X_train, y_train)


    y_pred = lgb.predict(X_val)


    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    mae_list_lgb.append(mae)
    rmse_list_lgb.append(rmse)

    print(f"LGBM MAE: {mae:.3f}, RMSE: {rmse:.3f}")


print("\n===== LightGBM Result =====")
print("Average MAE:", np.mean(mae_list_lgb))
print("Average RMSE:", np.mean(rmse_list_lgb))


===== LGBM Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000137 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2546
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 13.747439


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.560, RMSE: 0.746

===== LGBM Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000225 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2544
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.339485
LGBM MAE: 0.535, RMSE: 0.704

===== LGBM Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2552
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.603608


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.559, RMSE: 0.752

===== LGBM Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000223 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2556
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 15.157560
LGBM MAE: 0.556, RMSE: 0.739

===== LGBM Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2034
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000123 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2548
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.927446


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.555, RMSE: 0.738

===== LGBM Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2529
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.634021
LGBM MAE: 0.582, RMSE: 0.780

===== LGBM Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2540
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 15.099467
LGBM MAE: 0.601, RMSE: 0.790

===== LGBM Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
[LightGBM] [Info] Auto-choosing col-w

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [12]:
lgb_final = LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
lgb_final.fit(X_final_train, y_final_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000193 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2591
[LightGBM] [Info] Number of data points in the train set: 15841, number of used features: 12
[LightGBM] [Info] Start training from score 14.910369


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
              n_jobs=-1, random_state=42, subsample=0.8)

In [13]:
from sklearn.metrics import r2_score, max_error

y_pred = lgb_final.predict(X_final_test)

lgb_mae = mean_absolute_error(y_final_test, y_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
lgb_r2 = r2_score(y_final_test, y_pred)
lgb_max_err = max_error(y_final_test, y_pred)

print("\n===== LightGBM Test Results =====")
print(f"MAE: {lgb_mae:.3f}")
print(f"RMSE: {lgb_rmse:.3f}")
print(f"R2: {lgb_r2:.3f}")
print(f"Max Error: {lgb_max_err:.3f}")


===== LightGBM Test Results =====
MAE: 0.525
RMSE: 0.703
R2: 0.983
Max Error: 5.140


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
